# 02 · Prompt Engineering & Structured Output

In a multi-agent system, agents don't talk to *people*, they talk to *code*. The Planner's output is read by a
routing function that fans out researchers; the Critic's output decides whether the graph loops back. So every
"decision-making" agent must return **data with a guaranteed shape**, not free text.

This notebook covers the two halves of that problem:

1. **Prompt engineering**: how to write prompts that reliably produce good agent behaviour
   (`ChatPromptTemplate`, variables, roles, constraints, few-shot examples, injecting today's date).
2. **Structured output**: how to force and *validate* the shape of the answer
   (Pydantic schemas, `with_structured_output`, methods, validators, error recovery).

By the end we will have designed the **real `ResearchPlan` schema and Planner prompt** used in production.

## 0 · Setup

In [1]:
from nb_setup import setup, timer
ROOT = setup()

from langchain_openrouter import ChatOpenRouter

# Model roles decided in notebook 01 (timeout is in ms for ChatOpenRouter!)
reasoning_llm = ChatOpenRouter(model="nvidia/nemotron-3.5-super-vl:free", temperature=0, max_retries=1, timeout=90_000)
worker_llm    = ChatOpenRouter(model="baidu/ernie-5.5-flash-preview:free", temperature=0, max_retries=1, timeout=90_000)
reasoning_llm = reasoning_llm.with_fallbacks([worker_llm])  # resilience from day one

API keys:
  GROQ_API_KEY         MISSING
  OPENROUTER_API_KEY   set
  TAVILY_API_KEY       set
  LANGSMITH_API_KEY    set
  LangSmith tracing    true


LLM cache: data\llm_cache.sqlite


## 1 · Prompt templates

Hard-coding prompts with f-strings works for one call, but agents need prompts that are:
- **parameterised** (the question, today's date, the outline...),
- **role-structured** (system vs human messages),
- **reusable and testable** (we can render a template without calling a model).

`ChatPromptTemplate` gives us all three. `{variables}` are filled at invoke time; `.partial()` pre-fills values that
are the same for every call (like the date). A template is itself a **Runnable**, so it composes with a model using
the pipe operator: `prompt | model` (this is LCEL, the LangChain Expression Language).

In [2]:
from datetime import date
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert research assistant. Today's date is {today}."),
    ("human", "{question}"),
]).partial(today=date.today().isoformat())

# Render without calling any model: great for debugging and unit tests.
for m in prompt.invoke({"question": "What is retrieval-augmented generation?"}).to_messages():
    print(f"{m.type:>6}: {m.content}")

system: You are an expert research assistant. Today's date is 2026-11-18.
 human: What is retrieval-augmented generation?


### Why inject today's date?
Models have a training cutoff and **don't know what day it is**. A researcher asked about "the latest" developments
will otherwise search for stale years. Every agent prompt in the system will receive `{today}`.

## 2 · Anatomy of a good agent prompt

The prompts in this project follow a consistent structure, because each section removes a specific failure mode:

| Section | Purpose | Failure it prevents |
|---|---|---|
| **Role** | Who the agent is and its expertise | Generic, shallow answers |
| **Context** | Date, the overall objective, inputs from other agents | Stale or off-topic output |
| **Task** | One clear job, stated as steps | Agent doing other agents' jobs |
| **Constraints** | Limits: number of items, scope, what NOT to do | Overlong plans, overlapping sub-questions |
| **Output format** | Delegated to the schema (structured output) | Unparseable text |
| **Examples** (optional) | 1–2 demonstrations (few-shot) | Wrong granularity or style |

## 3 · Structured output with Pydantic

We describe the desired output as a **Pydantic model**. LangChain converts it into a JSON schema and sends it to
the model; the reply is parsed and **validated** back into a Python object.

**Important insight:** the class docstring and every `Field(description=...)` are *sent to the model*. They are
part of the prompt! Good descriptions are prompt engineering.

Below is the schema design for the Planner. Notice:
- `Literal[...]` restricts values to an enum. The router will use `search_strategy` to pick tools.
- `Field(min_length=..., max_length=...)` bounds the list. Too many sub-questions would mean too many parallel
  researchers and too many API calls.
- A `field_validator` adds checks JSON Schema can't express (no duplicate sub-questions).

In [3]:
from typing import Literal
from pydantic import BaseModel, Field, field_validator


class SubQuestion(BaseModel):
    """One focused, independently searchable research question."""
    question: str = Field(description="A specific question answerable via search, phrased as a search-friendly question")
    rationale: str = Field(description="Why this question is needed to answer the main objective (one sentence)")
    search_strategy: Literal["web", "academic", "encyclopedic"] = Field(
        description="web = news/blogs/docs, academic = research papers, encyclopedic = definitions/background")


class ResearchPlan(BaseModel):
    """A plan that decomposes a research query into focused sub-questions and a report outline."""
    objective: str = Field(description="The research objective restated in one precise sentence")
    research_type: Literal["factual", "comparative", "technical", "trend_analysis", "exploratory"]
    sub_questions: list[SubQuestion] = Field(min_length=3, max_length=5,
                                             description="3-5 non-overlapping sub-questions covering the objective")
    report_outline: list[str] = Field(min_length=3, max_length=7, description="Section headings for the final report")

    @field_validator("sub_questions")
    @classmethod
    def no_duplicates(cls, v: list[SubQuestion]) -> list[SubQuestion]:
        texts = [s.question.strip().lower() for s in v]
        if len(set(texts)) != len(texts):
            raise ValueError("sub_questions must be unique")
        return v


# This JSON schema is what the model actually receives:
import json
print(json.dumps(ResearchPlan.model_json_schema(), indent=1)[:1200], "...")

{
 "$defs": {
  "SubQuestion": {
   "description": "One focused, independently searchable research question.",
   "properties": {
    "question": {
     "description": "A specific question answerable via search, phrased as a search-friendly question",
     "title": "Question",
     "type": "string"
    },
    "rationale": {
     "description": "Why this question is needed to answer the main objective (one sentence)",
     "title": "Rationale",
     "type": "string"
    },
    "search_strategy": {
     "description": "web = news/blogs/docs, academic = research papers, encyclopedic = definitions/background",
     "enum": [
      "web",
      "academic",
      "encyclopedic"
     ],
     "title": "Search Strategy",
     "type": "string"
    }
   },
   "required": [
    "question",
    "rationale",
    "search_strategy"
   ],
   "title": "SubQuestion",
   "type": "object"
  }
 },
 "description": "A plan that decomposes a research query into focused sub-questions and a report outline.",
 "p

## 4 · The Planner prompt + structured output, end to end

`llm.with_structured_output(ResearchPlan)` returns a new Runnable whose output is a `ResearchPlan` instance.
Composing `planner_prompt | structured_llm` gives us a complete **Planner agent** in two lines.

Note on **methods** (how the schema is enforced):
- `function_calling` (default): the schema is sent as a *tool*, and the model "calls" it with the data. Widely supported.
- `json_schema`: uses the provider's native constrained decoding (guaranteed-valid JSON) when supported.

In [4]:
PLANNER_SYSTEM = """You are a senior research strategist. Today's date is {today}.

Your task: turn the user's research query into an actionable research plan.

Steps:
1. Restate the objective precisely and classify the research type.
2. Decompose it into 3-5 sub-questions that together fully answer the objective.
3. For each, choose the best source type (web, academic, encyclopedic).
4. Propose a logical outline for the final report.

Constraints:
- Sub-questions must NOT overlap; each must be answerable independently by a separate researcher.
- Prefer concrete, searchable phrasing over vague themes.
- For anything time-sensitive, mention the timeframe explicitly (relative to today's date)."""

planner_prompt = ChatPromptTemplate.from_messages([
    ("system", PLANNER_SYSTEM),
    ("human", "Research query: {query}"),
]).partial(today=date.today().isoformat())


# Does `.with_structured_output` work on a fallback wrapper (reasoning_llm = nemotron.with_fallbacks([ernie]))?
proxied = reasoning_llm.with_structured_output(ResearchPlan)
print(type(proxied).__name__, "->", [type(f).__name__ for f in proxied.fallbacks])

RunnableWithFallbacks -> ['RunnableSequence']


> **Subtle point: structure first, then fall back.** `reasoning_llm` is a `RunnableWithFallbacks`, not a chat model.
> Recent `langchain-core` versions *proxy* `with_structured_output` / `bind_tools` through the wrapper by inspecting
> type hints at runtime, so the cell above works with real models. But that proxy is implicit and fragile: while
> building this notebook it **crashed on LangChain's fake test models**, which we need for unit tests in Phase 3.
>
> So we use the explicit, test-friendly pattern: apply the schema to **each model first**, then chain the structured
> runnables with fallbacks. This shapes the production model factory: it exposes a helper that returns *structured*
> runnables with fallbacks.

In [5]:
nemotron = ChatOpenRouter(model="nvidia/nemotron-3.5-super-vl:free", temperature=0, max_retries=1, timeout=90_000)
ernie    = ChatOpenRouter(model="baidu/ernie-5.5-flash-preview:free", temperature=0, max_retries=1, timeout=90_000)


def structured(schema, primary, *fallbacks):
    """Structured-output runnable with fallbacks: apply the schema to each model, then chain them."""
    runnables = [m.with_structured_output(schema) for m in (primary, *fallbacks)]
    return runnables[0].with_fallbacks(runnables[1:]) if fallbacks else runnables[0]


planner = planner_prompt | structured(ResearchPlan, nemotron, ernie)

QUERY = "How are AI agents changing software engineering workflows in 2026?"
with timer("planner"):
    plan = planner.invoke({"query": QUERY})

print(type(plan).__name__, "|", plan.research_type)
print("Objective:", plan.objective, "\n")
for i, sq in enumerate(plan.sub_questions, 1):
    print(f"{i}. [{sq.search_strategy}] {sq.question}\n   why: {sq.rationale}")
print("\nOutline:", plan.report_outline)

[planner] 0.02s
ResearchPlan | trend_analysis
Objective: Analyze how AI agents are transforming software engineering workflows as of late 2026, including capabilities adopted, workflow changes across the SDLC, measured impacts, and adoption challenges. 

1. [web] Which AI agent capabilities (e.g., autonomous coding, debugging, code review, deployment management) are actively deployed in software engineering workflows in 2026?
   why: Establishes the baseline of what AI agents can actually do in software engineering today, distinguishing current deployment from hype or future roadmaps.
2. [web] How have specific software development lifecycle phases (requirements, coding, testing, CI/CD, incident response) been restructured around AI agent collaboration in 2026?
   why: Identifies concrete workflow changes at each stage of software delivery rather than general statements about AI assistance.
3. [web] What quantitative productivity, quality, and cycle-time impacts have been documented fr

## 5 · When structured output fails, and how to recover

Even good models occasionally return invalid data: missing fields, wrong enum values, too many items. Two tools:

1. **`include_raw=True`**: instead of raising, return `{"raw": AIMessage, "parsed": obj | None, "parsing_error": err | None}`.
   The graph can then *decide* what to do instead of crashing.
2. **Repair loop**: send the validation error back to the model and ask it to fix its own output. Models are very good
   at correcting a specific, explicit error message.

To demonstrate deterministically (without hoping the model fails), we take a deliberately broken output and run it
through validation, then ask the model to repair it.

In [6]:
from pydantic import ValidationError

broken_output = {
    "objective": "Understand AI agents in software engineering",
    "research_type": "survey",                       # not an allowed enum value
    "sub_questions": [                                # only 1 item, schema needs 3-5
        {"question": "What are AI coding agents?", "rationale": "background", "search_strategy": "google"}  # bad enum
    ],
    "report_outline": ["Intro", "Findings", "Conclusion"],
}

try:
    ResearchPlan.model_validate(broken_output)
except ValidationError as err:
    validation_error = err
    print(f"{err.error_count()} validation errors:\n")
    for e in err.errors():
        print(" -", ".".join(map(str, e["loc"])), "->", e["msg"])

2 validation errors:

 - research_type -> Input should be 'factual', 'comparative', 'technical', 'trend_analysis' or 'exploratory'
 - sub_questions.0.search_strategy -> Input should be 'web', 'academic' or 'encyclopedic'


Notice that only **2** errors are reported, although the list also has too few items. Pydantic validates the
*items* first and checks the list's `min_length` only once every item is valid. So validation errors can arrive in
**waves**: fixing one batch can reveal the next. That is why the repair step below re-validates its result
(via structured output), and why production code should allow a bounded number of repair attempts, not assume one
pass always succeeds.

In [7]:
REPAIR_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You fix JSON objects so they validate against a schema. Keep the original intent; change only what is needed."),
    ("human", "Original output:\n{output}\n\nValidation errors:\n{errors}\n\nOriginal task: {task}\n\nReturn the corrected object."),
])


def invoke_with_repair(chain_input: dict, broken: dict, error: ValidationError):
    """One repair attempt: show the model its own errors and ask for a corrected object."""
    repair_chain = REPAIR_PROMPT | structured(ResearchPlan, ernie, nemotron)
    return repair_chain.invoke({
        "output": json.dumps(broken, indent=1),
        "errors": str(error),
        "task": f"Research plan for: {chain_input['query']}",
    })


with timer("repair"):
    fixed = invoke_with_repair({"query": QUERY}, broken_output, validation_error)
print("repaired ->", fixed.research_type, "|", len(fixed.sub_questions), "sub-questions")
for sq in fixed.sub_questions:
    print(f"  [{sq.search_strategy}] {sq.question}")

[repair] 0.02s
repaired -> trend_analysis | 3 sub-questions
  [web] What are AI coding agents?
  [web] How are AI agents changing software engineering workflows in 2026?
  [academic] What are the benefits, risks, and limitations of AI agents in software engineering?


## 6 · Few-shot prompting: steering granularity

Zero-shot plans are often decent but **too broad** ("What is the impact of X?"). One worked example in the prompt
shows the model the *level of specificity* we want. We compare zero-shot vs one-shot on the same query using a cheap
heuristic: average sub-question length and whether sub-questions mention concrete entities/timeframes.

Few-shot examples cost input tokens on *every* call, so we only keep them if they visibly help.

In [8]:
FEW_SHOT_EXAMPLE = """Example of the expected granularity:
Query: "Is solid-state battery technology ready for electric vehicles?"
Good sub-questions:
- [academic] What energy density and cycle-life figures have solid-state cells achieved in peer-reviewed tests since 2024?
- [web] Which automakers have announced solid-state EV production timelines, and for which years?
- [web] What manufacturing bottlenecks currently limit solid-state cell production at scale?
Bad (too vague): "What are solid-state batteries?" or "What is the future of EVs?".
"""

few_shot_prompt = ChatPromptTemplate.from_messages([
    ("system", PLANNER_SYSTEM + "\n\n" + FEW_SHOT_EXAMPLE),
    ("human", "Research query: {query}"),
]).partial(today=date.today().isoformat())

few_shot_planner = few_shot_prompt | structured(ResearchPlan, nemotron, ernie)
with timer("few-shot planner"):
    plan_fs = few_shot_planner.invoke({"query": QUERY})

import re
def specificity(p: ResearchPlan) -> dict:
    qs = [s.question for s in p.sub_questions]
    concrete = sum(bool(re.search(r"\d{4}|[A-Z][a-z]+[A-Z]|%|benchmark|which|how many", q)) for q in qs)
    return {"n": len(qs), "avg_words": round(sum(len(q.split()) for q in qs) / len(qs), 1), "concrete": f"{concrete}/{len(qs)}"}

print("zero-shot:", specificity(plan))
print("one-shot :", specificity(plan_fs))
for sq in plan_fs.sub_questions:
    print(f"  [{sq.search_strategy}] {sq.question}")

[few-shot planner] 0.01s
zero-shot: {'n': 5, 'avg_words': 19.6, 'concrete': '5/5'}
one-shot : {'n': 4, 'avg_words': 22.2, 'concrete': '4/4'}
  [web] Which AI agent tools and capabilities (e.g., autonomous coding, debugging, testing, deployment) are actively used in software engineering workflows as of Q4 2026?
  [academic] What measurable changes in developer productivity, code quality, or delivery speed have been documented in peer-reviewed studies or industry reports comparing AI-agent-augmented versus traditional software teams in 2025-2026?
  [web] How have software team structures, code review processes, and CI/CD pipelines been reorganized to accommodate AI agent workflows in 2026?
  [web] What security, reliability, or governance challenges have emerged when deploying AI agents in software engineering workflows in 2026?


## 7 · Observations & Decisions

*Recorded from the run on 2026-11-18.*

**Observations**
- The Planner (Nemotron, ~12 s) produced a valid `ResearchPlan` first time: 5 non-overlapping sub-questions, sensible
  search strategies, explicit "2026" timeframes (thanks to `{today}` in the prompt).
- Pydantic `Field` descriptions and docstrings are sent to the model, so they act as prompt text.
- Fallback wrappers *can* proxy `with_structured_output` in this LangChain version, but the proxy is fragile (it broke
  on fake test models). Explicit "structure each model, then fall back" is the robust pattern.
- Validation errors arrive in **waves** (item errors before list-length errors). The repair pass fixed the broken plan
  in one attempt (Ernie, ~15 s), but repairs should be bounded loops, not single shots.
- **Few-shot did not help here:** zero-shot was already 5/5 "concrete" on our heuristic (avg 19.6 words); one-shot gave
  4/4 with longer questions (22.2 words). A strong instruction prompt already covers what the example teaches.
  (The heuristic is crude; notebook 08 will re-test this with an LLM judge.)

**Decisions**
1. All decision-making agents (Planner, Fact-checker, Critic) use **Pydantic structured output**.
2. Production model factory exposes `structured(schema, role)` → per-model structured runnables **with fallbacks**.
3. Structured calls get up to **2 automatic repair attempts** on `ValidationError`, then fail gracefully.
4. Prompts live in `src/research_system/prompts/` as templates with `{today}` injected. The Planner uses the
   **zero-shot** prompt (cheaper, equally specific); the few-shot variant stays available for the eval in notebook 08.
5. `ResearchPlan` / `SubQuestion` become `src/research_system/schemas/plan.py` (3–5 sub-questions caps fan-out cost).